# 02 — Dense and convolutional autoencoders (TensorFlow/Keras)

Modern companion to the original autoencoder Colab notebook. Datasets: MNIST and Fashion-MNIST, downloaded by Keras on first use.

## 1. Version and load MNIST

In [ ]:
import tensorflow as tf, numpy as np, matplotlib.pyplot as plt
print('TensorFlow',tf.__version__)
(x_train,_),(x_test,_)=tf.keras.datasets.mnist.load_data()
x_train=x_train.astype('float32')/255.; x_test=x_test.astype('float32')/255.
print(x_train.shape,x_test.shape,'range:',x_train.min(),x_train.max())

## 2. Dense autoencoder

An autoencoder minimizes reconstruction loss \(L(x,\\hat{x})\). The latent code is a compressed representation, but it is not automatically a deployable image codec: account for quantization, metadata and model size.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
inputs=keras.Input((784,)); x=layers.Dense(128,activation='relu')(inputs); x=layers.Dense(64,activation='relu')(x); z=layers.Dense(32,activation='relu',name='latent')(x); x=layers.Dense(64,activation='relu')(z); x=layers.Dense(128,activation='relu')(x); outputs=layers.Dense(784,activation='sigmoid')(x)
ae=keras.Model(inputs,outputs); encoder=keras.Model(inputs,z)
ae.compile(optimizer='adam',loss='binary_crossentropy')
ae.fit(x_train.reshape(-1,784),x_train.reshape(-1,784),validation_split=.1,epochs=15,batch_size=256)
recon=ae.predict(x_test.reshape(-1,784),verbose=0).reshape(-1,28,28)

## 3. Convolutional autoencoder for Fashion-MNIST

In [ ]:
(xf_train,_),(xf_test,_)=keras.datasets.fashion_mnist.load_data()
xf_train=xf_train.astype('float32')[...,None]/255.; xf_test=xf_test.astype('float32')[...,None]/255.
inputs=keras.Input((28,28,1)); x=layers.Conv2D(32,3,activation='relu',padding='same')(inputs); x=layers.MaxPooling2D(2,padding='same')(x); x=layers.Conv2D(16,3,activation='relu',padding='same')(x); z=layers.MaxPooling2D(2,padding='same',name='latent_map')(x); x=layers.Conv2D(16,3,activation='relu',padding='same')(z); x=layers.UpSampling2D(2)(x); x=layers.Conv2D(32,3,activation='relu',padding='same')(x); x=layers.UpSampling2D(2)(x); outputs=layers.Conv2D(1,3,activation='sigmoid',padding='same')(x)
cae=keras.Model(inputs,outputs); cae.compile(optimizer='adam',loss='binary_crossentropy'); cae.summary()
cae.fit(xf_train,xf_train,validation_split=.1,epochs=15,batch_size=256)
xf_recon=cae.predict(xf_test[:16],verbose=0)

## 4. Visualize reconstructions

In [ ]:
fig,ax=plt.subplots(2,8,figsize=(14,4))
for i in range(8):
 ax[0,i].imshow(xf_test[i,:,:,0],cmap='gray'); ax[0,i].axis('off')
 ax[1,i].imshow(xf_recon[i,:,:,0],cmap='gray'); ax[1,i].axis('off')
plt.tight_layout()

## 5. Image compression formulas and tips

- Raw grayscale image bits: \(B_{raw}=H\\times W\\times b\), where \(b\) is bits per pixel.
- Quantized latent bits (simple estimate): \(B_z=N_z\\times q\), where \(N_z\) is latent scalar count and \(q\) is bits per latent value.
- Approximate ratio: \(R=B_{raw}/(B_z+B_{metadata}+B_{model/share})\).
- Use SSIM and PSNR in addition to MSE; inspect perceptual artifacts.
- Use nearest-neighbour resizing for categorical masks, never bilinear interpolation.

## Exercises
Vary latent size, add Gaussian noise, and compare MSE/SSIM and actual serialized sizes.